In [1]:
from pyspark.sql import SparkSession
import random
from faker import Faker

# Initialize Spark and Faker
spark = SparkSession.builder \
    .appName("NikeOrdersDataGeneration") \
    .getOrCreate()

fake = Faker()

# Constants and settings
NUM_RECORDS = 1000000
USER_IDS = list(range(1, NUM_RECORDS + 1))



# Function to generate sample orders using RDD
def generate_orders():
    def create_order(record_id):
        return (
            record_id,
            random.choice(USER_IDS),
            fake.date_this_year().strftime("%Y-%m-%d %H:%M:%S"),
            round(random.uniform(20, 500), 2),
            record_id, 
            record_id,
            record_id,
            record_id
        )

    orders_rdd = spark.sparkContext.parallelize(range(1, NUM_RECORDS + 1)).map(create_order)
    return orders_rdd.toDF(['order_id', 'user_id', 'order_date', 'total_amount', 'discount_id', 'payment_transaction_id', 'shipping_id', 'order_status_id'])

# Function to generate order items using RDD
def generate_order_items():
    def create_order_item(record_id):
        return (
            record_id,
            record_id,
            f"P{random.randint(1000, 9999)}",
            random.randint(1, 10),
            round(random.uniform(5, 150), 2)
        )

    order_items_rdd = spark.sparkContext.parallelize(range(1, NUM_RECORDS + 1)).map(create_order_item)
    return order_items_rdd.toDF(['order_item_id', 'order_id', 'product_id', 'quantity', 'price'])

# Function to generate addresses using RDD
def generate_addresses():
    def create_address(record_id):
        return (
            record_id,
            record_id,
            random.choice(['billing', 'shipping']),
            fake.street_address(),
            fake.secondary_address(),
            fake.city(),
            fake.state(),
            fake.zipcode(),
            fake.country(),
            fake.phone_number()
        )

    addresses_rdd = spark.sparkContext.parallelize(range(1, NUM_RECORDS + 1)).map(create_address)
    return addresses_rdd.toDF(['address_id', 'order_id', 'address_type', 'address_line1', 'address_line2', 'city', 'state', 'postal_code', 'country', 'phone_number'])

# Function to generate shipping details using RDD
def generate_shipping_details():
    def create_shipping_detail(record_id):
        return (
            record_id,
            record_id,
            record_id,
            fake.random_element(['Standard', 'Express', 'Overnight']),
            fake.random_element(['pending', 'in transit', 'delivered', 'returned']),
            fake.uuid4()
        )

    shipping_details_rdd = spark.sparkContext.parallelize(range(1, NUM_RECORDS + 1)).map(create_shipping_detail)
    return shipping_details_rdd.toDF(['shipping_id', 'address_id', 'order_id', 'shipping_method', 'shipping_status', 'tracking_number'])

# Function to generate payment transactions using RDD
def generate_payment_transactions():
    def create_payment_transaction(record_id):
        return (
            record_id,
            record_id,
            random.choice(['credit card', 'debit card', 'paypal', 'gift card', 'voucher']),
            round(random.uniform(20, 500), 2),
            fake.date_this_year().strftime("%Y-%m-%d %H:%M:%S"),
            fake.uuid4()
        )

    payment_transactions_rdd = spark.sparkContext.parallelize(range(1, NUM_RECORDS + 1)).map(create_payment_transaction)
    return payment_transactions_rdd.toDF(['payment_transaction_id', 'order_id', 'payment_method', 'payment_amount', 'payment_date', 'transaction_reference'])

# Function to generate order statuses using RDD
def generate_order_status():
    def create_order_status(record_id):
        return (
            record_id,
            record_id,
            random.choice(['pending', 'confirmed', 'shipped', 'delivered', 'cancelled', 'returned']),
            fake.date_this_year().strftime("%Y-%m-%d %H:%M:%S")
        )

    order_statuses_rdd = spark.sparkContext.parallelize(range(1, NUM_RECORDS + 1)).map(create_order_status)
    return order_statuses_rdd.toDF(['status_id', 'order_id', 'status', 'status_change_date'])

# Function to generate discounts using RDD
def generate_discounts():
    def create_discount(record_id):
        return (
            record_id,
            fake.lexify(text='DISCOUNT????'),
            record_id,
            random.choice(['percentage', 'fixed']),
            round(random.uniform(5, 50), 2),
            fake.date_this_year().strftime("%Y-%m-%d %H:%M:%S"),
            fake.date_this_year().strftime("%Y-%m-%d %H:%M:%S")
        )

    discounts_rdd = spark.sparkContext.parallelize(range(1, NUM_RECORDS + 1)).map(create_discount)
    return discounts_rdd.toDF(['discount_id', 'discount_code', 'order_id', 'discount_type', 'discount_value', 'start_date', 'end_date'])

# Generate all data
orders_df = generate_orders()
order_items_df = generate_order_items()
addresses_df = generate_addresses()
shipping_details_df = generate_shipping_details()
payment_transactions_df = generate_payment_transactions()
order_status_df = generate_order_status()
discounts_df = generate_discounts()

# Showing a small sample of the generated data
orders_df.show(5)
order_items_df.show(5)
addresses_df.show(5)
shipping_details_df.show(5)
payment_transactions_df.show(5)
order_status_df.show(5)
discounts_df.show(5)

+--------+-------+-------------------+------------+-----------+----------------------+-----------+---------------+
|order_id|user_id|         order_date|total_amount|discount_id|payment_transaction_id|shipping_id|order_status_id|
+--------+-------+-------------------+------------+-----------+----------------------+-----------+---------------+
|       1| 340121|2024-02-05 00:00:00|      176.64|          1|                     1|          1|              1|
|       2| 257855|2024-01-08 00:00:00|       87.64|          2|                     2|          2|              2|
|       3| 638642|2024-11-23 00:00:00|      359.66|          3|                     3|          3|              3|
|       4| 913640|2024-06-20 00:00:00|       72.64|          4|                     4|          4|              4|
|       5| 746219|2024-08-20 00:00:00|      181.24|          5|                     5|          5|              5|
+--------+-------+-------------------+------------+-----------+-----------------

In [2]:
import os
# Set a path for  directory
local_folder_path = "./orders"

# Ensure the directory exists, or create it
if not os.path.exists(local_folder_path):
    os.makedirs(local_folder_path)


# Coalesce the data to reduce the number of output files (e.g., coalesce to 1 file for small datasets)

num_partitions = 1  

orders_df = orders_df.coalesce(num_partitions)
order_items_df = order_items_df.coalesce(num_partitions)
addresses_df = addresses_df.coalesce(num_partitions)
shipping_details_df = shipping_details_df.coalesce(num_partitions)
payment_transactions_df = payment_transactions_df.coalesce(num_partitions)
order_status_df = order_status_df.coalesce(num_partitions)
discounts_df = shipping_details_df.coalesce(num_partitions)


# Write DataFrames to Parquet in the local folder using Snappy compression (default in Spark)
orders_df.write \
    .mode("overwrite") \
    .parquet(local_folder_path + "/orders/", compression="snappy")

order_items_df.write \
    .mode("overwrite") \
    .parquet(local_folder_path + "/orderitems/", compression="snappy")

addresses_df.write \
    .mode("overwrite") \
    .parquet(local_folder_path + "/addresses/", compression="snappy")

shipping_details_df.write \
    .mode("overwrite") \
    .parquet(local_folder_path + "/shippingdetails/", compression="snappy")

payment_transactions_df.write \
    .mode("overwrite") \
    .parquet(local_folder_path + "/paymenttransactions/", compression="snappy")

order_status_df.write \
    .mode("overwrite") \
    .parquet(local_folder_path + "/orderstatus/", compression="snappy")

discounts_df.write \
    .mode("overwrite") \
    .parquet(local_folder_path + "/discounts/", compression="snappy")


print("Data successfully written to local folder in Parquet format with Snappy compression.")

Data successfully written to local folder in Parquet format with Snappy compression.
